In [1]:
import os, sys
os.chdir('..')
sys.path.append(os.getcwd())
print('Working directory:', os.getcwd())

Working directory: /Users/lr/policy_vector/shareable


# Mean-Difference Vector Evaluation

Load the pre-generated dataset and mean-difference vector, then evaluate
projection separation on a limited subset (change `limit` to run the full set).

In [2]:
import json
from pathlib import Path
from policy_vector_pipeline import load_dataset, MeanDifferenceVector, ActivationCollector
from scripts.evaluate_vector import compute_stats, load_model
from transformers import logging
logging.set_verbosity_error()

dataset_path = Path('data/on_policy_persona.json')
vector_path = Path('artifacts/qwen3_onpolicy_mean.pt')
model_name = 'Qwen/Qwen3-4B'

dataset = load_dataset(dataset_path)
vector = MeanDifferenceVector.load(vector_path)
layer_ids = sorted(vector.layer_vectors.keys())
print('Examples:', len(dataset.examples), 'Layers:', layer_ids[:5], '...')

Examples: 60 Layers: [18, 19, 20, 21, 22] ...


In [3]:
model, tokenizer = load_model(model_name, device_map='auto', dtype='auto')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
collector = ActivationCollector(
    model,
    tokenizer,
    layers=layer_ids,
    reduction='mean',
    response_only=True,
)
activations = collector.collect_dataset(dataset, progress=True, limit=1)  # Set to None to process full dataset

In [ ]:
stats = compute_stats(activations, vector)
stats[:3]

In [ ]:
top_layer = stats[0]['layer']
vec = vector.layer_vectors[top_layer].to(torch.float32)
vec /= torch.linalg.norm(vec) + 1e-8
proj_on = torch.stack(activations['on'][top_layer]).to(torch.float32) @ vec
proj_off = torch.stack(activations['off'][top_layer]).to(torch.float32) @ vec
proj_on, proj_off